# Further Performance Metrics

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
X_train = pd.read_csv("/content/drive/My Drive/Colab Notebooks/data/X_train_final_smote.csv")
X_test = pd.read_csv("/content/drive/My Drive/Colab Notebooks/data/X_test_final.csv")
X_val = pd.read_csv("/content/drive/My Drive/Colab Notebooks/data/X_val_final.csv")

y_train = pd.read_csv("/content/drive/My Drive/Colab Notebooks/data/y_train_final_smote.csv")
y_test = pd.read_csv("/content/drive/My Drive/Colab Notebooks/data/y_test_final.csv")
y_val = pd.read_csv("/content/drive/My Drive/Colab Notebooks/data/y_val_final.csv")

## 5 Combinations of Features

In [4]:
feature_sets = {
    "set_1": X_train.columns.tolist(),

    "set_2": [
        "sact",
        "physio1",
        "safety5",
        "belonging1",
        "esteem3",
        "trust3",
        "ethical80",
        "civic53",
        "compassion1",
        "growth16",
        "age",
        "marital_status",
        "religious1",
        "income_scale"
    ],

    "set_3": [
        "sact",
        "physio1",
        "safety5",
        "belonging1",
        "esteem3"
    ],

    "set_4": [
        "trust3",
        "ethical80",
        "civic53",
        "compassion1",
        "growth16"
    ],

    "set_5": [
        "sact",
        "physio1",
        "safety5",
        "trust3",
        "ethical80"
    ]
}

Create data splits.

In [5]:
data_splits = {}

for name, features in feature_sets.items():
    data_splits[name] = {
        "X_train": X_train[features],
        "X_val": X_val[features],
        "X_test": X_test[features],
    }

In [6]:
import warnings
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")
pd.pandas.set_option("display.max_columns", None)

In [7]:
models = [
    {
        "name": "RandomForest",
        "estimator": RandomForestClassifier(random_state=42),
        "param_grid": {
            "n_estimators": [50, 100, 150],
            "max_depth": [None, 8, 12],
            "min_samples_split": [2, 4, 6],
            "max_features": ['sqrt', 'log2']
        }
    },
    {
        "name": "XGBoost",
        "estimator": XGBClassifier(eval_metric='mlogloss', random_state=42),
        "param_grid": {
            "n_estimators": [50, 100, 150],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [3, 4, 5]
        }
    },
    {
        "name": "LogisticRegression",
        "estimator": LogisticRegression(random_state=42, max_iter=3000),
        "param_grid": {
            "penalty": ['l1', 'l2'],
            "C": [0.01, 0.1, 1, 10],
            "solver": ['liblinear', 'saga']
    }
}
]


## Model Training

In [8]:
import sys
sys.path.append('/content/drive/My Drive/Colab Notebooks/')
import my_tools

all_results = {}
all_models = {}

for combo_name, data in data_splits.items():
    print(f"\n==============================")
    print(f"Running feature set: {combo_name}")
    print(f"==============================")

    results_df, best_models =  my_tools.train_ensemble_models(
        data["X_train"], y_train,
        data["X_val"], y_val,
        data["X_test"], y_test,
        models
    )

    all_results[combo_name] = results_df
    all_models[combo_name] = best_models


Running feature set: set_1
y_train: Converted to 1D pd automatically...
y_test: Converted to 1D pd automatically...
y_val: Converted to 1D pd automatically...

Training RandomForest...

Training XGBoost...

Training LogisticRegression...

Running feature set: set_2
y_train: Converted to 1D pd automatically...
y_test: Converted to 1D pd automatically...
y_val: Converted to 1D pd automatically...

Training RandomForest...

Training XGBoost...

Training LogisticRegression...

Running feature set: set_3
y_train: Converted to 1D pd automatically...
y_test: Converted to 1D pd automatically...
y_val: Converted to 1D pd automatically...

Training RandomForest...

Training XGBoost...

Training LogisticRegression...

Running feature set: set_4
y_train: Converted to 1D pd automatically...
y_test: Converted to 1D pd automatically...
y_val: Converted to 1D pd automatically...

Training RandomForest...

Training XGBoost...

Training LogisticRegression...

Running feature set: set_5
y_train: Convert

In [9]:
with open("/content/drive/My Drive/Colab Notebooks/data/model_feature_comparison.txt", "w") as f:
    for combo_name, df in all_results.items():
        f.write("\n" + "="*60 + "\n")
        f.write(f"FEATURE SET: {combo_name}, Predictors: {len(feature_sets[combo_name])}\n")
        f.write("="*60 + "\n\n")

        # ===== Metrics table =====
        f.write(df.to_string(index=False))
        f.write("\n\n")

        # ===== Classification Reports =====
        f.write("Classification Reports:\n")

        X_test_subset = data_splits[combo_name]["X_test"]

        for model_name, model_info in all_models[combo_name].items():
            model = model_info["model"] # extract trained model

            y_pred = model.predict(X_test_subset)

            f.write(f"\n\n--- {model_name} ---\n")
            f.write(classification_report(y_test, y_pred, digits=4))

        f.write("\n\n")

    print("INFO: Output generated successfully!")